# Strict Final Holdout Evaluation (Issue #37) — RUN ONCE

⚠️ **This notebook produces the final thesis number. It must be run exactly once,**
after the tuning-stage winner is frozen.

Protocol (mirrors the original grouped-split test evaluation):
1. Refit the selected model on **train + validation** (selection is complete, so the
   validation split has served its purpose — same rule as the original workflow).
2. Predict the untouched `test_strict.csv` once.
3. Save metrics, predictions, and full provenance metadata.

A guard cell refuses to run if holdout artifacts already exist. **Do not delete them to
rerun with a different config** — that would invalidate the holdout claim. If a rerun is
ever genuinely needed (e.g. a bug in the pipeline), document why in
`docs/DESIGN_DECISIONS.md` first.

## 1. Imports and guard

In [ ]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "datasets").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.strict_funnel import euro_metrics, fit_and_predict
from src.strict_protocol import COMPONENT_GROUP_COLUMN, add_component_group
from src.tree_modeling import TARGET_COLUMN, load_training_data

HOLDOUT_DIR = ROOT / "artifacts" / "strict_final_holdout"
if HOLDOUT_DIR.exists() and any(HOLDOUT_DIR.iterdir()):
    raise SystemExit(
        "Holdout artifacts already exist in artifacts/strict_final_holdout/. "
        "The final evaluation has been run; it must not be repeated."
    )

## 2. Confirm the frozen winner

Set `CONFIRMED_MODEL` to the tuning-stage winner, and flip
`I_CONFIRM_SELECTION_IS_FROZEN` to `True`. Both are deliberate manual steps: the holdout
must never run on autopilot.

In [ ]:
CONFIRMED_MODEL = None  # e.g. "random_forest"
I_CONFIRM_SELECTION_IS_FROZEN = False

if not I_CONFIRM_SELECTION_IS_FROZEN or CONFIRMED_MODEL is None:
    raise SystemExit(
        "Set CONFIRMED_MODEL and I_CONFIRM_SELECTION_IS_FROZEN = True after freezing "
        "the winner from the tuning stage."
    )

summary_path = ROOT / "artifacts/strict_model_tuning" / CONFIRMED_MODEL / "best_tuning_summary.json"
best_summary = json.loads(summary_path.read_text(encoding="utf-8"))
features = list(best_summary["feature_names"])
config = dict(best_summary["config"])
print("Winner:", CONFIRMED_MODEL, "|", best_summary["feature_variant"], "|", best_summary["config_name"])

## 3. Refit on train + validation, evaluate once on the strict test split

In [ ]:
prepared = load_training_data(
    ROOT / "datasets/splits_strict/train_strict.csv",
    ROOT / "datasets/splits_strict/validation_strict.csv",
)
test_df = pd.read_csv(ROOT / "datasets/splits_strict/test_strict.csv")
# Recompute date offsets for the test frame with the SAME reference date as the fit data.
for column in ("first_seen_date", "last_seen_date", "scrape_date"):
    if column in test_df.columns:
        test_df[column] = pd.to_datetime(test_df[column], errors="coerce")
if prepared.reference_first_seen_date is not None and "first_seen_date" in test_df.columns:
    reference = pd.Timestamp(prepared.reference_first_seen_date)
    test_df["first_seen_day_offset"] = (test_df["first_seen_date"] - reference).dt.days
    test_df["last_seen_day_offset"] = (test_df["last_seen_date"] - reference).dt.days
    test_df["listing_midpoint_day_offset"] = (
        test_df["first_seen_day_offset"] + test_df["last_seen_day_offset"]
    ) / 2

fit_df = add_component_group(
    pd.concat([prepared.train_df, prepared.validation_df], ignore_index=True)
)
print("Fit rows (train+validation):", len(fit_df), "| Test rows:", len(test_df))

predictions = fit_and_predict(
    CONFIRMED_MODEL, fit_df, test_df, features, config, COMPONENT_GROUP_COLUMN
)
metrics = euro_metrics(test_df[TARGET_COLUMN], predictions, prefix="test")
metrics

## 4. Freeze the holdout artifacts

In [ ]:
HOLDOUT_DIR.mkdir(parents=True, exist_ok=True)

final_summary = {
    "generated_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
    "issue": "#37 final strict holdout evaluation",
    "model_type": CONFIRMED_MODEL,
    "evaluation_split": "datasets/splits_strict/test_strict.csv",
    "fit_scope": "train_strict + validation_strict (selection complete)",
    "split_provenance": "datasets/splits_strict/strict_split_summary.json (seed 32)",
    "selection_source": str(summary_path.relative_to(ROOT)),
    "feature_variant": best_summary.get("feature_variant"),
    "config_name": best_summary.get("config_name"),
    "target_mode": best_summary.get("target_mode"),
    "feature_count": len(features),
    "feature_names": features,
    "config": config,
    **metrics,
}
(HOLDOUT_DIR / "final_holdout_metrics.json").write_text(
    json.dumps(final_summary, indent=2, default=str) + "\n", encoding="utf-8"
)
pd.DataFrame(
    {"actual_price": test_df[TARGET_COLUMN].to_numpy(), "predicted_price": predictions}
).to_csv(HOLDOUT_DIR / "final_holdout_predictions.csv", index=False)

print("Frozen holdout artifacts written to:", HOLDOUT_DIR)
print(json.dumps(metrics, indent=2))

## 5. After this

- Record the result and the date in `docs/DESIGN_DECISIONS.md` and the roadmap.
- The number in `final_holdout_metrics.json` is the thesis headline result — whatever it is.
- Export the deployment bundle separately (`scripts/export_random_forest_model.py`);
  deployment fitting is not part of this evaluation artifact.